# 🧙‍♂️ Магические методы (часть 3): менеджеры контекста, `__new__` и `__del__`

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Управление ресурсами, создание объектов и… прощание</h2>
  <p>Сегодня мы познакомимся с тремя важными магическими методами, которые управляют жизненным циклом объектов:</p>
  <ul>
    <li><code>__enter__</code> / <code>__exit__</code> — для работы с контекстными менеджерами (блоки <code>with</code>)</li>
    <li><code>__new__</code> — настоящий конструктор (создание объекта)</li>
    <li><code>__del__</code> — финализатор (прощание с объектом)</li>
  </ul>
  <p>Эти методы позволяют писать более безопасный, эффективный и «питоничный» код.</p>
</div>

## 🎯 Цели лекции

- Понять, зачем нужны менеджеры контекста и как реализовать свой собственный с помощью `__enter__` и `__exit__`
- Разобраться, чем `__new__` отличается от `__init__` и в каких случаях его используют
- Узнать о существовании `__del__` и почему его лучше избегать
- Научиться применять эти методы на практике

## 📚 План

1. **Менеджеры контекста (`__enter__`, `__exit__`)**  
   - Проблема управления ресурсами  
   - Синтаксис `with`  
   - Реализация своего контекстного менеджера  
   - Обработка исключений в `__exit__`  
   - Примеры: таймер, блокировка, работа с файлами

2. **Настоящий конструктор `__new__`**  
   - Чем отличается от `__init__`  
   - Создание объекта  
   - Паттерн Singleton  
   - Неизменяемые объекты (наследование от встроенных типов)

3. **Финализатор `__del__`**  
   - Когда вызывается  
   - Почему не стоит полагаться на него для освобождения ресурсов  
   - Альтернативы (контекстные менеджеры)

4. **Итоги и ключевые моменты**

## 🧾 Менеджеры контекста (`__enter__`, `__exit__`)

### Зачем нужны?
Часто мы работаем с ресурсами, которые нужно обязательно освобождать после использования: файлы, сетевые соединения, блокировки, транзакции БД. Забыть закрыть файл или освободить блокировку — частая ошибка.

В Python придумали элегантное решение — менеджеры контекста и оператор `with`. Он гарантирует, что ресурс будет освобождён даже в случае исключения.

### Как это работает?
Объект, который можно использовать в `with`, должен реализовывать два магических метода:
- `__enter__(self)` — вызывается при входе в блок `with`. Возвращает объект, который будет доступен через `as`.
- `__exit__(self, exc_type, exc_val, exc_tb)` — вызывается при выходе из блока (в том числе при исключении). Параметры описывают возникшее исключение (если его не было, все три равны `None`). Если метод возвращает `True`, исключение подавляется; если `False` (или ничего) — исключение пробрасывается дальше.

### Пример: свой менеджер для работы с файлом
(хотя встроенный `open` уже является менеджером)

In [ ]:
# Пример простого контекстного менеджера для файла
print("📁 Создаём свой контекстный менеджер для файла")

class ManagedFile:
    def __init__(self, filename, mode='r'):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        print(f"Открываем файл {self.filename}")
        self.file = open(self.filename, self.mode, encoding='utf-8')
        return self.file  # то, что попадёт в переменную after 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            print(f"Закрываем файл {self.filename}")
            self.file.close()
        # Если было исключение, сообщим о нём
        if exc_type is not None:
            print(f"Поймали исключение: {exc_val}")
        # Возвращаем False, чтобы исключение пробросилось (если не хотим подавлять)
        return False  # можно не писать, тогда автоматически None -> False

# Использование
print("\n--- Нормальная работа ---")
with ManagedFile('test.txt', 'w') as f:
    f.write('Привет, мир!')

print("\n--- Работа с исключением ---")
try:
    with ManagedFile('test.txt', 'r') as f:
        content = f.read()
        print(f"Содержимое: {content}")
        raise ValueError("Что-то пошло не так")
except ValueError:
    print("Исключение обработано снаружи")

# Убедимся, что файл действительно закрыт
print("\nФайл закрыт автоматически, можем его удалить")
import os
os.remove('test.txt')

### Разбор `__exit__` параметров
- `exc_type` — класс исключения (например, `ValueError`)
- `exc_val` — объект исключения
- `exc_tb` — traceback (объект с информацией о стеке)

Если нужно **подавить** исключение (сделать вид, что его не было), верните `True`. Осторожно: это может скрыть ошибки!

### Пример: контекстный менеджер-таймер
Полезно для измерения времени выполнения блока кода.

In [ ]:
import time

class Timer:
    """Контекстный менеджер для измерения времени."""
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # можем вернуть сам объект, чтобы получить доступ к длительности

    def __exit__(self, *args):
        self.end = time.perf_counter()
        self.duration = self.end - self.start
        print(f"⏱️ Время выполнения: {self.duration:.6f} сек")

# Использование
print("Измеряем время:")
with Timer():
    sum(i**2 for i in range(10**6))

### Пример: блокировка для потоков (упрощённо)
Менеджеры контекста идеально подходят для синхронизации: захватили блокировку вошёл, освободили при выходе.

In [ ]:
import threading

class LockContext:
    def __init__(self, lock):
        self.lock = lock

    def __enter__(self):
        print("Захватываем блокировку")
        self.lock.acquire()
        return self

    def __exit__(self, *args):
        print("Освобождаем блокировку")
        self.lock.release()

# Демонстрация
my_lock = threading.Lock()
with LockContext(my_lock):
    print("Критическая секция")

## 🏗️ Настоящий конструктор `__new__`

### Отличие от `__init__`
- `__new__` — **статический метод** (на самом деле, особый), который создаёт и возвращает новый экземпляр класса.
- `__init__` — инициализирует уже созданный объект.

Обычно мы не переопределяем `__new__`, но есть случаи, когда это необходимо:
- Создание неизменяемых объектов (например, наследование от `int`, `str`, `tuple`)
- Реализация паттерна **Singleton** (класс, у которого может быть только один экземпляр)
- Иногда для управления созданием объекта (например, пул объектов)

### Синтаксис
```python
class MyClass:
    def __new__(cls, *args, **kwargs):
        # cls — сам класс
        obj = super().__new__(cls)  # делегируем создание родителю
        # можно модифицировать obj перед возвратом
        return obj

    def __init__(self, ...):
        # инициализация
        ...

In [ ]:
# Пример: Singleton (одиночка)
print("👤 Реализуем Singleton через __new__")

class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("Создаём единственный экземпляр")
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value=None):
        # __init__ будет вызываться каждый раз при Singleton(...)
        # поэтому нужно защититься от повторной инициализации
        if not hasattr(self, 'initialized'):
            self.value = value
            self.initialized = True
            print(f"Инициализация со значением {value}")
        else:
            print(f"Повторный вызов __init__, значение остаётся {self.value}")

# Проверка
a = Singleton(10)
b = Singleton(20)
print(f"a is b? {a is b}")  # True
print(f"a.value = {a.value}, b.value = {b.value}")  # оба 10

### Пример: неизменяемый класс на основе tuple
Если мы хотим создать свой класс, похожий на кортеж, нужно переопределить `__new__`, потому что кортеж неизменяем и создаётся сразу целиком.

In [ ]:
class Point(tuple):
    """Точка на плоскости, наследует tuple."""
    def __new__(cls, x, y):
        # Создаём кортеж из двух элементов и возвращаем его как экземпляр Point
        return super().__new__(cls, (x, y))

    # Можно добавить свойства для удобства
    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]

# Использование
p = Point(3, 4)
print(f"Точка: {p}, x={p.x}, y={p.y}")
print(f"Тип: {type(p)}")  # <class '__main__.Point'>

## 🗑️ Финализатор `__del__`

### Что это?
`__del__(self)` — метод, который вызывается, когда объект уничтожается сборщиком мусора (когда количество ссылок на объект становится равным нулю).

### Почему его не любят?
1. **Непредсказуемое время вызова** — нельзя точно знать, когда сборщик мусора сработает (в CPython — сразу при обнулении счётчика ссылок, но есть циклы ссылок, тогда срабатывает цикличный GC с задержкой).
2. **Может не вызваться вовсе** — если программа завершается, интерпретатор может просто не вызывать `__del__` для всех объектов.
3. **Исключения в `__del__`** — они игнорируются и пишутся в stderr.
4. **Проблемы с циклическими ссылками** — объекты с `__del__` сложнее собираются сборщиком.

**Вывод:** не используйте `__del__` для освобождения важных ресурсов (файлы, соединения). Вместо этого применяйте контекстные менеджеры или явные методы закрытия (`.close()`).

In [ ]:
# Пример (чисто для демонстрации, в реальном коде так лучше не делать!)
print("⚠️ Демонстрация __del__")

class Resource:
    def __init__(self, name):
        self.name = name
        print(f"Ресурс {self.name} создан")

    def __del__(self):
        print(f"Ресурс {self.name} уничтожен")

# Создаём и удаляем
r = Resource("файл")
del r  # удаляем явно — __del__ вызовется сразу
print("После удаления")

# А теперь пример, когда __del__ может не вызваться
r2 = Resource("сеть")
r2 = None  # удаляем ссылку — объект становится недоступным, __del__ должен вызваться
import gc
gc.collect()  # принудительный сбор мусора (для надёжности)
print("Конец программы")

### Правильный подход: использовать контекстный менеджер
Вместо `__del__` реализуйте метод `close()` и поддерживайте протокол контекстного менеджера.

In [ ]:
class BetterResource:
    def __init__(self, name):
        self.name = name
        self.opened = True
        print(f"Ресурс {self.name} открыт")

    def close(self):
        if self.opened:
            print(f"Ресурс {self.name} закрыт")
            self.opened = False

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

# Использование
with BetterResource("база данных") as res:
    print("Работаем с ресурсом")
# автоматически закроется при выходе из блока

## 🔍 Итоги и ключевые моменты

### Менеджеры контекста (`__enter__` / `__exit__`)
- Используются для гарантированного освобождения ресурсов.
- Работают с оператором `with`.
- `__exit__` получает информацию об исключении и может его подавить.
- Примеры: работа с файлами, блокировки, таймеры, транзакции.

### `__new__`
- Создаёт новый экземпляр класса (вызывается до `__init__`).
- Применяется для реализации Singleton, создания неизменяемых объектов, кастомизации создания.
- Всегда возвращает объект (обычно результат вызова `super().__new__`).
- В большинстве случаев не нужен — пользуйтесь `__init__`.

### `__del__`
- Финализатор, вызывается при уничтожении объекта.
- **Ненадёжен** — не стоит использовать для освобождения ресурсов.
- Лучшие альтернативы: контекстные менеджеры, явный вызов `close()`.

### 🧠 Проверь себя

**Вопрос 1:** Что произойдёт, если в методе `__exit__` вернуть `True`?

**Вопрос 2:** Почему в Singleton нужно проверять `hasattr(self, 'initialized')` в `__init__`?

**Вопрос 3:** В каком порядке вызываются `__new__` и `__init__` при создании объекта?

**Вопрос 4:** Можно ли использовать `__del__` для автоматического закрытия файла? Почему это плохая идея?

<details>
<summary>Ответы</summary>

1. Исключение, возникшее в блоке `with`, будет подавлено (не пробросится дальше).
2. Потому что `__init__` вызывается каждый раз при вызове класса (`Singleton()`), даже если экземпляр уже существует. Нам нужно инициализировать объект только один раз.
3. Сначала `__new__` (создаёт объект), затем `__init__` (инициализирует).
4. Технически можно, но из-за непредсказуемого времени вызова и возможных проблем с циклическими ссылками это ненадёжно. Лучше использовать `with`.

</details>

---

## 📖 Домашнее задание

1. **Контекстный менеджер для временной смены директории**  
   Реализуйте класс `cd`, который при входе в блок `with` меняет текущую рабочую директорию на указанную, а при выходе возвращает обратно. Используйте `os.chdir()`.

2. **Пул объектов через `__new__`**  
   Создайте класс `ConnectionPool`, который выдаёт ограниченное количество соединений (объектов). При запросе нового экземпляра либо возвращается существующий свободный, либо создаётся новый (если не превышен лимит). Для простоты можно хранить объекты в списке.

3. **Сравнение `__del__` и контекстного менеджера**  
   Напишите два класса: `BadResource` с `__del__` и `GoodResource` с поддержкой `with`. Продемонстрируйте, что происходит при исключении внутри блока и как себя ведёт каждый из подходов.

---